In [ ]:
from dotenv import load_dotenv()
load_dotenv()

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings,ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate

In [ ]:
loader = PyPDFLoader('../data/data_science_syllabus.pdf')
docs = loader.load()
len(docs)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splitted_data = splitter.split_documents(docs)
len(splitted_data)


In [ ]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

In [ ]:
vector_store = Chroma.from_documents(
  documents= splitted_data,
  embedding=embeddings
)

In [ ]:
query = "Machine Learnings and Data Science Content"
data = vector_store.similarity_search(query = query)

In [ ]:
context = ''
for doc in data:
  context += doc.page_content + '\n'
print(context)

In [ ]:
llm = ChatOpenAI (model='gpt-5')


In [ ]:
# res = llm.invoke(f"Can you provide me the answer based on provided context fro my question,context:{context} and question:{query}")

def get_context(query:str):
  data = vector_store.similarity_search(query=query)
  context = ''
  for doc in data:
   context += doc.page_content + '\n'

  return {
    "context":context,
    "Question": query
  }

In [ ]:
prompt =PromptTemplate.from_template("""You are a helpful assistant and provide answerd based on the context for user question. and 
    if you don't know the answer, then you can say that 'I dont know.'
    Context: {context}
    Question: {question} """)

In [ ]:
rag_chain = get_context| prompt | llm

In [ ]:
res = rag_chain.invoke('What is duration of my course/')

print(res.content)